# 4. 概率融合、模型比较与列线图

将训练集的影像和临床**折外预测概率**输入 Logistic 融合。`fusion_data` 按 ID 严格对齐，还会核对结局与折编号。

不能用测试标签重新拟合融合模型。基础模型 OOF 数据参与融合拟合后，也不能再把融合在这些数据上的表现视为独立验证。

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import ov_radiomics as rad

# Run notebooks in order, with examples/ as the working directory.
output = Path("output/notebook_demo")

In [ ]:
parts = {
    "Rad": joblib.load(output / "rad" / "training.joblib"),
    "Clinical": joblib.load(output / "clinical" / "training.joblib"),
}
oof = rad.fusion_data(parts, model="LR", cohort="oof")
fusion = rad.fit_fusion(oof, target="y_true", score_columns=["Rad", "Clinical"])
joblib.dump(fusion, output / "fusion.joblib")
oof.head()

## 固定测试集预测
沿用训练好的基础模型和融合模型，下面没有任何 `fit(test)`。

In [ ]:
test = rad.fusion_data(parts, model="LR", cohort="test")
test["Fusion"] = fusion.predict_proba(test)
scores = test[["Rad", "Clinical", "Fusion"]]
test.to_csv(output / "test_probabilities.csv", index=False)
metrics = pd.DataFrame({name: rad.binary_metrics(test.y_true, scores[name], threshold=0.5) for name in scores}).T
metrics.to_csv(output / "test_metrics.csv")
metrics

In [ ]:
rad.plot_roc(test.y_true, scores)
rad.plot_calibration(test.y_true, scores, n_bins=5)
curves = rad.decision_curve(test.y_true, scores, thresholds=np.linspace(.01,.99,99))
rad.plot_dca(curves, ylim=(-.1,.65))
plt.show()

## 配对 AUC 比较与校准检验
DeLong 接收同一批独立患者的配对预测，返回普通 p 值。H–L 分组遇到重复概率时可能减少实际组数，并返回对应自由度与分组表；p 值大不代表已证明校准良好。

In [ ]:
delong = pd.DataFrame({name: rad.delong_test(test.y_true, test.Fusion, test[name]) for name in ["Rad","Clinical"]}).T
delong.to_csv(output / "delong.csv")
delong

In [ ]:
hl = rad.hosmer_lemeshow(test.y_true, test.Fusion, n_bins=5)
print({key: value for key, value in hl.items() if key != "table"})
hl["table"]

In [ ]:
rad.plot_prediction_scores(test.y_true, test.Fusion)
rad.plot_confusion_matrix(test.y_true, test.Fusion)
figure = rad.plot_nomogram(fusion, risk_ticks=[.1,.25,.5,.75,.9], figsize=(10,6))
figure.savefig(output / "nomogram.png", dpi=150, bbox_inches="tight")
plt.show()

列线图直接来自已拟合 LR 的系数，预测变量是两个基础模型的阳性概率。它不会重新训练模型。

此四步教程全部基于合成数据；完成执行表示功能链路可运行，不构成临床验证。